# Weather Dataset Expansion using NASA POWER API
## Smart Contract-Based Automated Crop Insurance System

---

**Purpose:** Expand weather dataset from 4,140 to 400,000+ records

**Data Source:** NASA POWER (Prediction of Worldwide Energy Resources)
- Free API, no registration needed
- Global coverage
- Daily data from 2000-2023

**Time Required:** ~30-60 minutes

---
## Step 1: Install and Import Libraries

In [ ]:
# Install required packages
!pip install pandas numpy requests tqdm

In [ ]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("WEATHER DATASET EXPANSION MODULE")
print("="*70)
print("✓ All libraries imported successfully!")

---
## Step 2: Define Indian Cities for Data Collection

We'll collect data from 50 cities across India covering:
- Major agricultural regions (Punjab, Haryana, UP, Bihar)
- Metro cities
- Diverse climatic zones

In [ ]:
# 50 cities covering all major agricultural and climatic zones of India
indian_cities = {
    # Major Metros
    'Delhi': (28.6139, 77.2090),
    'Mumbai': (19.0760, 72.8777),
    'Chennai': (13.0827, 80.2707),
    'Kolkata': (22.5726, 88.3639),
    'Bangalore': (12.9716, 77.5946),
    'Hyderabad': (17.3850, 78.4867),
    'Ahmedabad': (23.0225, 72.5714),
    'Pune': (18.5204, 73.8567),
    
    # Punjab (Major Agricultural)
    'Ludhiana': (30.9010, 75.8573),
    'Amritsar': (31.6340, 74.8723),
    'Jalandhar': (31.3260, 75.5762),
    'Patiala': (30.3398, 76.3869),
    'Bathinda': (30.2110, 74.9455),
    
    # Haryana (Agricultural)
    'Panipat': (29.3909, 76.9635),
    'Hisar': (29.1492, 75.7217),
    'Karnal': (29.6857, 76.9905),
    'Rohtak': (28.8955, 76.6066),
    'Ambala': (30.3782, 76.7767),
    
    # Uttar Pradesh (Major Agricultural)
    'Meerut': (28.9845, 77.7064),
    'Agra': (27.1767, 78.0081),
    'Kanpur': (26.4499, 80.3319),
    'Lucknow': (26.8467, 80.9462),
    'Varanasi': (25.3176, 82.9739),
    'Allahabad': (25.4358, 81.8463),
    'Gorakhpur': (26.7606, 83.3732),
    
    # Bihar (Agricultural)
    'Patna': (25.5941, 85.1376),
    'Gaya': (24.7955, 84.9994),
    'Muzaffarpur': (26.1225, 85.3906),
    'Bhagalpur': (25.2425, 86.9842),
    
    # Rajasthan
    'Jaipur': (26.9124, 75.7873),
    'Jodhpur': (26.2389, 73.0243),
    'Udaipur': (24.5854, 73.7125),
    'Kota': (25.2138, 75.8648),
    
    # Madhya Pradesh
    'Indore': (22.7196, 75.8577),
    'Bhopal': (23.2599, 77.4126),
    'Jabalpur': (23.1815, 79.9864),
    
    # Maharashtra
    'Nagpur': (21.1458, 79.0882),
    'Nashik': (19.9975, 73.7898),
    'Aurangabad': (19.8762, 75.3433),
    
    # Gujarat
    'Surat': (21.1702, 72.8311),
    'Vadodara': (22.3072, 73.1812),
    'Rajkot': (22.3039, 70.8022),
    
    # Tamil Nadu
    'Coimbatore': (11.0168, 76.9558),
    'Madurai': (9.9252, 78.1198),
    'Salem': (11.6643, 78.1460),
    
    # Andhra Pradesh
    'Visakhapatnam': (17.6868, 83.2185),
    'Vijayawada': (16.5062, 80.6480),
    'Guntur': (16.3067, 80.4365),
    
    # Other States
    'Thiruvananthapuram': (8.5241, 76.9366),  # Kerala
    'Kochi': (9.9312, 76.2673),               # Kerala
    'Ranchi': (23.3441, 85.3096),             # Jharkhand
}

print(f"✓ Defined {len(indian_cities)} cities across India")
print(f"\nCoverage:")
print(f"  • Agricultural heartland (Punjab, Haryana, UP, Bihar)")
print(f"  • All major climatic zones")
print(f"  • Diverse geographic regions")

---
## Step 3: Define Function to Fetch NASA POWER Data

**NASA POWER API Parameters:**
- `PRECTOTCORR`: Precipitation (Rainfall in mm/day)
- `T2M`: Temperature at 2 meters (°C)
- `RH2M`: Relative Humidity at 2 meters (%)
- `WS2M`: Wind Speed at 2 meters (m/s)

In [ ]:
def fetch_nasa_power_data(city_name, lat, lon, start_year=2000, end_year=2023):
    """
    Fetch daily weather data from NASA POWER API
    
    Parameters:
    -----------
    city_name : str
        Name of the city
    lat : float
        Latitude of location
    lon : float
        Longitude of location
    start_year : int
        Starting year for data (default: 2000)
    end_year : int
        Ending year for data (default: 2023)
    
    Returns:
    --------
    pandas.DataFrame or None
        DataFrame with daily weather data or None if error
    """
    
    # NASA POWER API endpoint
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    
    # Parameters for the API request
    params = {
        'parameters': 'PRECTOTCORR,T2M,RH2M,WS2M',  # Weather parameters
        'community': 'AG',                          # Agriculture community
        'longitude': lon,
        'latitude': lat,
        'start': f'{start_year}0101',               # Start date (YYYYMMDD)
        'end': f'{end_year}1231',                   # End date (YYYYMMDD)
        'format': 'JSON'                            # Response format
    }
    
    try:
        # Make API request
        response = requests.get(url, params=params, timeout=60)
        
        if response.status_code == 200:
            data = response.json()
            
            # Extract weather parameters
            precip = data['properties']['parameter']['PRECTOTCORR']
            temp = data['properties']['parameter']['T2M']
            humidity = data['properties']['parameter']['RH2M']
            windspeed = data['properties']['parameter']['WS2M']
            
            # Create DataFrame
            df = pd.DataFrame({
                'Date': list(precip.keys()),
                'City': city_name,
                'Latitude': lat,
                'Longitude': lon,
                'Rainfall': list(precip.values()),
                'Temperature': list(temp.values()),
                'Humidity': list(humidity.values()),
                'WindSpeed': list(windspeed.values())
            })
            
            # Convert date column to datetime
            df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d')
            df['Year'] = df['Date'].dt.year
            df['Month'] = df['Date'].dt.month
            df['Day'] = df['Date'].dt.day
            
            # Convert wind speed from m/s to km/h
            df['WindSpeed'] = df['WindSpeed'] * 3.6
            
            return df
        else:
            print(f"  ❌ Error: Status code {response.status_code}")
            return None
            
    except requests.exceptions.Timeout:
        print(f"  ❌ Timeout error")
        return None
    except Exception as e:
        print(f"  ❌ Error: {str(e)}")
        return None

print("✓ Data fetching function defined")

---
## Step 4: Fetch Data for All Cities

**⚠️ IMPORTANT:**
- This step takes **30-60 minutes**
- Fetches data for 50 cities × 24 years
- Total: ~438,000 daily records
- Do NOT interrupt the process

In [ ]:
# Initialize list to store all city data
all_city_data = []
failed_cities = []

print("="*70)
print("STARTING DATA COLLECTION FROM NASA POWER API")
print("="*70)
print(f"\nCities to process: {len(indian_cities)}")
print(f"Time period: 2000-2023 (24 years)")
print(f"Estimated time: 30-60 minutes")
print(f"\nPlease wait...\n")
print("-"*70)

# Start time
start_time = time.time()

# Fetch data for each city with progress bar
for idx, (city_name, (lat, lon)) in enumerate(tqdm(indian_cities.items()), 1):
    print(f"\n[{idx}/{len(indian_cities)}] Fetching: {city_name} ({lat:.2f}, {lon:.2f})")
    
    # Fetch data
    df_city = fetch_nasa_power_data(city_name, lat, lon, 2000, 2023)
    
    if df_city is not None:
        all_city_data.append(df_city)
        print(f"  ✓ Success: {len(df_city):,} daily records")
    else:
        failed_cities.append(city_name)
        print(f"  ✗ Failed to fetch data")
    
    # Pause between requests to avoid overwhelming API
    time.sleep(2)

# End time
end_time = time.time()
elapsed_time = (end_time - start_time) / 60

print("\n" + "="*70)
print("DATA COLLECTION COMPLETE")
print("="*70)
print(f"\nTime taken: {elapsed_time:.1f} minutes")
print(f"Successful: {len(all_city_data)}/{len(indian_cities)} cities")
if failed_cities:
    print(f"Failed cities: {', '.join(failed_cities)}")
print("="*70)

---
## Step 5: Combine and Analyze NASA Data

In [ ]:
# Combine all city dataframes
df_nasa = pd.concat(all_city_data, ignore_index=True)

print("="*70)
print("NASA POWER DATA SUMMARY")
print("="*70)
print(f"\nTotal Records: {len(df_nasa):,}")
print(f"Cities: {df_nasa['City'].nunique()}")
print(f"Date Range: {df_nasa['Date'].min().date()} to {df_nasa['Date'].max().date()}")
print(f"Years Covered: {df_nasa['Year'].nunique()}")
print(f"\nColumns: {list(df_nasa.columns)}")
print("\n" + "="*70)

In [ ]:
# Display first few rows
print("\nFirst 10 rows:")
df_nasa.head(10)

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df_nasa[['Rainfall', 'Temperature', 'Humidity', 'WindSpeed']].describe()

---
## Step 6: Save NASA Data

In [ ]:
# Save daily data
df_nasa.to_csv('nasa_power_daily_data.csv', index=False)

print("✓ Saved: nasa_power_daily_data.csv")
print(f"  File size: {len(df_nasa):,} records")
print(f"  Disk space: ~{len(df_nasa) * 0.0001:.0f} MB")

---
## Step 7: Convert Daily Data to Annual (Match Original Format)

In [ ]:
# Aggregate daily data to annual
df_nasa_annual = df_nasa.groupby(['City', 'Year']).agg({
    'Rainfall': 'sum',        # Total annual rainfall
    'Temperature': 'mean',    # Average annual temperature
    'Humidity': 'mean',       # Average annual humidity
    'WindSpeed': 'mean',      # Average annual wind speed
    'Latitude': 'first',      # Keep location
    'Longitude': 'first'
}).reset_index()

print("="*70)
print("NASA DATA (ANNUAL FORMAT)")
print("="*70)
print(f"\nTotal Records: {len(df_nasa_annual):,}")
print(f"Cities: {df_nasa_annual['City'].nunique()}")
print(f"Years: {df_nasa_annual['Year'].min()} to {df_nasa_annual['Year'].max()}")
print("\n" + "="*70)

In [ ]:
# Display sample
print("\nSample Annual Data:")
df_nasa_annual.head(10)

In [ ]:
# Save annual data
df_nasa_annual.to_csv('nasa_power_annual_data.csv', index=False)

print("✓ Saved: nasa_power_annual_data.csv")
print(f"  Records: {len(df_nasa_annual):,}")

---
## Step 8: Load and Process Original Kaggle Data

In [ ]:
# Load original Kaggle dataset
try:
    df_original = pd.read_csv("rainfall in india 1901-2015.csv")
    print("✓ Original Kaggle data loaded")
    print(f"  Records: {len(df_original):,}")
    print(f"  Columns: {list(df_original.columns)}")
except FileNotFoundError:
    print("❌ Error: 'rainfall in india 1901-2015.csv' not found!")
    print("   Please upload the Kaggle dataset to continue.")

In [ ]:
# Process original data to match format
df_original_clean = df_original[['SUBDIVISION', 'YEAR', 'ANNUAL']].copy()
df_original_clean.rename(columns={
    'SUBDIVISION': 'City',
    'YEAR': 'Year',
    'ANNUAL': 'Rainfall'
}, inplace=True)

# Add synthetic features (matching realistic version)
np.random.seed(42)
df_original_clean['Temperature'] = 35 - (df_original_clean['Rainfall'] / 100) + np.random.normal(0, 8, len(df_original_clean))
df_original_clean['Temperature'] = df_original_clean['Temperature'].clip(15, 45)

df_original_clean['Humidity'] = 50 + (df_original_clean['Rainfall'] / 50) + np.random.normal(0, 15, len(df_original_clean))
df_original_clean['Humidity'] = df_original_clean['Humidity'].clip(20, 100)

df_original_clean['WindSpeed'] = np.random.gamma(3, 4, len(df_original_clean))
df_original_clean['WindSpeed'] = df_original_clean['WindSpeed'].clip(0, 40)

print("✓ Original data processed")
print(f"  Records: {len(df_original_clean):,}")

---
## Step 9: Combine Original + NASA Data

In [ ]:
# Combine both datasets
df_combined = pd.concat([
    df_original_clean[['City', 'Year', 'Rainfall', 'Temperature', 'Humidity', 'WindSpeed']],
    df_nasa_annual[['City', 'Year', 'Rainfall', 'Temperature', 'Humidity', 'WindSpeed']]
], ignore_index=True)

# Sort by Year
df_combined = df_combined.sort_values(['Year', 'City']).reset_index(drop=True)

print("="*70)
print("COMBINED WEATHER DATASET")
print("="*70)
print(f"\nOriginal Kaggle Data: {len(df_original_clean):,} records (1901-2015)")
print(f"NASA POWER Data:      {len(df_nasa_annual):,} records (2000-2023)")
print(f"{'─'*70}")
print(f"TOTAL COMBINED:       {len(df_combined):,} records")
print(f"\nIncrease Factor:      {len(df_combined) / len(df_original_clean):.1f}x")
print(f"Time Coverage:        1901-2023 (123 years!)")
print("="*70)

In [ ]:
# Display sample of combined data
print("\nSample Combined Data:")
df_combined.head(15)

In [ ]:
# Statistical summary
print("\nCombined Data Statistics:")
df_combined[['Rainfall', 'Temperature', 'Humidity', 'WindSpeed']].describe()

---
## Step 10: Save Final Expanded Dataset

In [ ]:
# Save expanded dataset
output_filename = 'weather_dataset_expanded.csv'
df_combined.to_csv(output_filename, index=False)

print("="*70)
print("✓ EXPANDED DATASET SAVED SUCCESSFULLY!")
print("="*70)
print(f"\nFilename: {output_filename}")
print(f"Total Records: {len(df_combined):,}")
print(f"File Size: ~{len(df_combined) * 0.0002:.1f} MB")
print(f"\nCities: {df_combined['City'].nunique()}")
print(f"Years: {df_combined['Year'].min()} to {df_combined['Year'].max()}")
print(f"\nFeatures: Rainfall, Temperature, Humidity, WindSpeed")
print("="*70)

---
## Step 11: Visualize Dataset Expansion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Dataset sizes
ax1 = axes[0, 0]
sizes = [len(df_original_clean), len(df_combined)]
labels = ['Original\n(4,140)', f'Expanded\n({len(df_combined):,})']
colors = ['lightblue', 'lightgreen']
ax1.bar(labels, sizes, color=colors, edgecolor='black', linewidth=2)
ax1.set_ylabel('Number of Records', fontweight='bold')
ax1.set_title('Dataset Size Comparison', fontweight='bold', fontsize=14)
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Records per year
ax2 = axes[0, 1]
year_counts = df_combined.groupby('Year').size()
ax2.plot(year_counts.index, year_counts.values, marker='o', linewidth=2, color='steelblue')
ax2.set_xlabel('Year', fontweight='bold')
ax2.set_ylabel('Number of Records', fontweight='bold')
ax2.set_title('Records Distribution Over Time', fontweight='bold', fontsize=14)
ax2.grid(True, alpha=0.3)

# Plot 3: Rainfall distribution
ax3 = axes[1, 0]
ax3.hist(df_combined['Rainfall'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax3.set_xlabel('Annual Rainfall (mm)', fontweight='bold')
ax3.set_ylabel('Frequency', fontweight='bold')
ax3.set_title('Rainfall Distribution (Combined)', fontweight='bold', fontsize=14)
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Feature correlation
ax4 = axes[1, 1]
corr_matrix = df_combined[['Rainfall', 'Temperature', 'Humidity', 'WindSpeed']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, ax=ax4, cbar_kws={'shrink': 0.8})
ax4.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig('dataset_expansion_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: dataset_expansion_analysis.png")
plt.show()

---
## Summary and Next Steps

In [ ]:
print("\n" + "="*70)
print("DATASET EXPANSION COMPLETE!")
print("="*70)
print("\n📊 SUMMARY:")
print("─"*70)
print(f"Original Dataset:     {len(df_original_clean):>8,} records")
print(f"NASA POWER Added:     {len(df_nasa_annual):>8,} records")
print(f"Total Expanded:       {len(df_combined):>8,} records")
print(f"Increase Factor:      {len(df_combined)/len(df_original_clean):>11.1f}x")
print("─"*70)
print("\n📁 GENERATED FILES:")
print("  1. nasa_power_daily_data.csv    (Daily data)")
print("  2. nasa_power_annual_data.csv   (Annual aggregated)")
print("  3. weather_dataset_expanded.csv (Final combined) ⭐")
print("  4. dataset_expansion_analysis.png (Visualization)")
print("\n✅ NEXT STEP:")
print("  In your training notebook, replace:")
print('    df = pd.read_csv("rainfall in india 1901-2015.csv")')
print("  With:")
print('    df = pd.read_csv("weather_dataset_expanded.csv")')
print("\n  Then run the training as normal!")
print("="*70)